In [1]:
from Scripts.pywin32_postinstall import verbose
!pip install langchain langchain-text-splitters langchain-community

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 25.1 MB/s  0:00:00

   -------- ------------------------------- 1/5 [pydantic-settings]
   -------- ------------------------------- 1/5 [pydantic-settings]
   -------- ------------------------------- 1/5 [pydantic-settings]
   ---------------- ----------------------- 2/5 [langchain-text-splitters]
   ---------------- ----------------------- 2/5 [langchain-text-splitters]
   ------------------------ --------------- 3/5 [langchain-classic]
   ------------------------ --------------- 3/5 [langchain-classic]
   ------------------------ --------------- 3/5 [langchain-classic]
   ------------------------ --------------- 3/5 [langchain-classic]
   ------------------------ --------------- 3/5 [langchain-classic]
   ----------------

In [38]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [13]:
from langchain.agents import create_agent
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_core.tools import tool

data = """
LlamaIndex is an open-source data orchestration framework designed to bridge the gap between Large Language Models (LLMs) and your custom or private data. In essence, it helps developers integrate diverse data sources like documents, databases, APIs, and more with powerful LLMs so the models can generate answers based on specific data rather than only general knowledge. This makes LLM-based applications more context-aware and useful for real-world use cases.

Core Functionality and Workflow

At the heart of LlamaIndex is its Retrieval-Augmented Generation (RAG) pipeline. The process typically involves ingesting raw data, breaking it into structured chunks, converting those chunks into vector embeddings, and storing them in a vector database for efficient semantic search. When a query is made, LlamaIndex retrieves the most relevant data pieces and supplies them as context to the LLM to generate more grounded, accurate responses.

Features and Capabilities

LlamaIndex offers a flexible set of tools for developers including connectors to multiple data formats (PDFs, databases, cloud storage, etc.), abstraction layers for building query engines and chatbots, and support for AI agents that can perform complex multi-step tasks. It has SDKs available in both Python and TypeScript, making it adaptable across different tech stacks.

Use Cases and Applications

Developers and businesses use LlamaIndex to build a variety of AI-powered applications such as semantic search engines, intelligent chatbots, document summarization tools, recommendation systems, and enterprise knowledge assistants. By enabling context-aware querying of large and unstructured data, it enhances the ability of LLMs to deliver more accurate responses tailored to specific domains.
"""

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore(embeddings)

doc = Document(page_content=data)

# Index chunks
vector_store.add_documents(documents=[doc])

@tool()
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    return retrieved_docs

tools = [retrieve_context]

prompt = (
    "You have access to a tool that retrieves context"
    "Use the tool to help answer user queries."
)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

agent = create_agent(llm, tools, system_prompt=prompt)

response = agent.invoke({"messages": "tell me about llama index in 2 lines"})

print("AI AGENT RESPONSE")

print(response)



[Document(id='4859ee76-4131-417e-82bc-0007f2788aa1', metadata={}, page_content='\nLlamaIndex is an open-source data orchestration framework designed to bridge the gap between Large Language Models (LLMs) and your custom or private data. In essence, it helps developers integrate diverse data sources like documents, databases, APIs, and more with powerful LLMs so the models can generate answers based on specific data rather than only general knowledge. This makes LLM-based applications more context-aware and useful for real-world use cases.\n\nCore Functionality and Workflow\n\nAt the heart of LlamaIndex is its Retrieval-Augmented Generation (RAG) pipeline. The process typically involves ingesting raw data, breaking it into structured chunks, converting those chunks into vector embeddings, and storing them in a vector database for efficient semantic search. When a query is made, LlamaIndex retrieves the most relevant data pieces and supplies them as context to the LLM to generate more gr

In [35]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("../../data/source")
doc=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=25)
chunks=splitter.split_documents(doc)

embeddings=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001" )

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=doc)

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

agent=create_agent(llm,tools,system_prompt=prompt)

response=agent.invoke({"messages": "tell me about llama index in 2 lines"})
print("AI Meta Information")
print(response)

print("AI Response:")
print(response["messages"][-1].content)

# response=agent.invoke({"messages": "what is my previous query ?"})
# print("AI Response:")
# print(response["messages"][-1].content)

AI Response:
[{'type': 'text', 'text': 'I do not have access to previous queries. I am a large language model, trained by Google, and I am designed to respond to your current input.', 'extras': {'signature': 'CvICAb4+9vu9YB9N0e5C9YE99KyLIrJwfQ+AwwR5PzIrsoqh6lG3TLlTApRFfjk7GGP7p+YPfeWwZldh6yLwWTm37aC2jG75WTC/3EB3BtxFpGjnjKFR5Ty60z87IlcinW1JfnUCPhNihE+97+KkvYPvQKuUi/Z+86yrYOUoeNRvIN6rW3xWmqmKQIB9BmG9PaR0KJbCQCdjX9JTwqTcnzCIuWblJLk42r6oycifOOCeqgixRUR/vIA20bQ+YgEhy+j1BEh6i5b3+OKvaFU3E3PUSmVxrGKTxUt6sjRoHQjQZGjomIKxjDXE6wrMZhg7MpIwhodrD+er5xCLgHmjZ8W9fW2h9BhB6aZiavSW24nOamnTqRMi+QoM5DUkmgre9//5yn608wu/2H3zrV0iWYjZLrdUBc8QG1aim4uWIWdfjnHX0+AHnON8VochVn+PykQ4Qhgoi/K+DSEICYs0xC55x1o22gqMcfSQZt1N+PVU5BB3Tw=='}}]


In [46]:
api_key = os.getenv("PINE_CONE_KEY")
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)
index = pc.Index("documents")

vector_store = PineconeVectorStore(embedding=embeddings, index=index)
vector_store.add_documents(documents=chunks)

@tool
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    print(retrieved_docs)
    return retrieved_docs

tools=[retrieve_context]
agent=create_agent(llm,system_prompt=prompt,tools=tools)

response=agent.invoke({"messages": "tell me about llama index in 2 lines"})
print("AI Meta Information")
print(response)

print("AI Response:")
print(response["messages"][-1].content)


[Document(id='c2813c74-0738-4c18-912d-8f9b8bc0debd', metadata={'source': '../data/source'}, page_content='When a query is made, LlamaIndex retrieves the most relevant data pieces and supplies them as'), Document(id='31935ce7-9bed-456a-92f3-5fab66196385', metadata={'source': '../data/source'}, page_content='LlamaIndex is an open-source data orchestration framework designed to bridge the gap between Large')]
AI Meta Information
{'messages': [HumanMessage(content='tell me about llama index in 2 lines', additional_kwargs={}, response_metadata={}, id='88771984-99cb-4ff7-9722-f6a939257ee8'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'retrieve_context', 'arguments': '{"query": "LlamaIndex"}'}, '__gemini_function_call_thought_signatures__': {'0c365af4-080f-492e-a198-37dff1b9a4d4': 'CpACAb4+9vtN1/YRALXFARW2lUrE3qM8eZCGHIzWnxZ48nPzlbcOmIDHWWh6refud1DOngi9L7432R3MXP43o3tM/la1WA4BfmU1/ql40hnwpqFQww/kw10VRmcdHCD2XZVNzVtGDVv+bDvN+TtA3GjXqDZM4bfaAOFF68cPPy3h/FeYYcRg0B+HYOKYhb

In [42]:
!pip install -qU langchain-qdrant

In [48]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import os
from langchain_community.document_loaders import PyPDFLoader


# ---------------------------
# 2. Connect to LOCAL Qdrant server
# ---------------------------
# IMPORTANT:
# Use host/port, not ":memory:", if you want browser dashboard visualization
client = QdrantClient(host="localhost", port=6333)

collection_name = "test"

# ---------------------------
# 3. Create collection if missing
# ---------------------------
vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE
        )
    )

# ---------------------------
# 4. Create LangChain vector store
# ---------------------------
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

# ---------------------------
# 5. Add sample documents
# ---------------------------

loader = PyPDFLoader("../../data/clean_code.pdf")
documents = loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=25)
chunks=splitter.split_documents(documents)

vector_store.add_documents(chunks)

print("Documents added successfully.\n")

# ---------------------------
# 6. Run a similarity search
# ---------------------------
results = vector_store.similarity_search("What is LlamaIndex?", k=3)

print("Search Results:")
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)

# ---------------------------
# 7. Optional: inspect points directly from Qdrant
# ---------------------------
points, next_page = client.scroll(
    collection_name=collection_name,
    limit=10,
    with_payload=True,
    with_vectors=False
)

print("\nStored Points in Qdrant:")
for p in points:
    print(f"\nID: {p.id}")
    print("Payload:", p.payload)

print("\nOpen dashboard at: http://localhost:6333/dashboard")

Documents added successfully.

Search Results:

Result 1
Content: LlamaIndex is a framework for building LLM applications with data.
Metadata: {'source': 'doc1', 'topic': 'llm', '_id': '64dc9baa-6a33-4502-9209-f1a560e00a07', '_collection_name': 'test'}

Result 2
Content: LangChain helps build applications using language models and tools.
Metadata: {'source': 'doc2', 'topic': 'llm', '_id': '357c3f44-61f2-46d5-953a-bde2acb2f962', '_collection_name': 'test'}

Result 3
Content: Qdrant is a vector database for similarity search and retrieval.
Metadata: {'source': 'doc3', 'topic': 'vector-db', '_id': '708fa153-324a-42b9-a7d6-84b6217e3e85', '_collection_name': 'test'}

Stored Points in Qdrant:

ID: 0d473831-f400-4c1c-9b8d-8565c6246553
Payload: {'page_content': '32 Chapter 3: Functions\nConsider the code in Listing 3-1. It’ s hard to ﬁnd a long function in FitNesse, 1 but\nafter a bit of searching I came across this one. Not only is it long, but it’ s got duplicated\ncode, lots of odd strings,

In [50]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(llm, checkpointer=InMemorySaver())

result = agent.invoke({"messages": "Hi, My name is Bob"},{"configurable": {"thread_id": "1"}})
print(result)
result2 = agent.invoke({"messages": "what is my name?"},{"configurable": {"thread_id": "1"}})
print(result2)

{'messages': [HumanMessage(content='Hi, My name is Bob', additional_kwargs={}, response_metadata={}, id='ea254184-2097-471d-a782-9cc8931caf32'), AIMessage(content="Hi Bob! It's nice to meet you.\n\nHow can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cc2c6-9e4f-7ea2-9dc2-f5defbe9c887-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 137, 'total_tokens': 144, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 118}})]}
{'messages': [HumanMessage(content='Hi, My name is Bob', additional_kwargs={}, response_metadata={}, id='ea254184-2097-471d-a782-9cc8931caf32'), AIMessage(content="Hi Bob! It's nice to meet you.\n\nHow can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_prov